In [ ]:
import marimo as mo

# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression.

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:

Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task
  - `MultiClassMLP` for the multi-class classification task
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept).
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already.
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

---
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.

In [ ]:
NAMES = "group11-section1"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

### Cost Function
Binary classification uses binary cross-entropy (log loss), which penalizes differences between predicted probabilities and binary labels. Multi-class classification uses categorical cross-entropy, which measures divergence between predicted probability distributions across classes and one-hot encoded targets. Regression uses mean squared error (MSE), which penalizes the squared continuous distance between predicted values and real-valued targets.

### Output Layer Changes
Binary classification requires 1 output unit with a sigmoid activation function to output probabilities bounded between 0 and 1. Multi-class classification requires as many output units as there are classes (K units) with a softmax activation function to produce a valid probability distribution that sums to 1. Regression requires 1 output unit with no activation function (identity/linear) so the network can predict unconstrained real values.

### Data Standardization
Standardizing features to zero mean and unit variance ensures that features with larger original scales do not disproportionately dominate gradient updates. This stabilizes training, prevents vanishing or exploding gradients, and leads to faster, more uniform parameter convergence.

### Overfitting Detection
Overfitting is detected by monitoring both training and holdout metrics across epochs. If training loss continues to decrease and training accuracy improves while holdout loss starts increasing or holdout performance degrades, the model has begun memorizing training noise rather than generalizing.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
import pandas as pd
import torch
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

_X, _y = make_moons(n_samples=1000, noise=0.2, random_state=42)
_X_train, _X_test, _y_train, _y_test = train_test_split(
    _X, _y, test_size=0.2, random_state=42
)

_scaler = StandardScaler()
_X_train_scaled = _scaler.fit_transform(_X_train)
_X_test_scaled = _scaler.transform(_X_test)

_X_train_t = torch.tensor(_X_train_scaled, dtype=torch.float32)
_y_train_t = torch.tensor(_y_train, dtype=torch.float32).unsqueeze(1)
_X_test_t = torch.tensor(_X_test_scaled, dtype=torch.float32)
_y_test_t = torch.tensor(_y_test, dtype=torch.float32).unsqueeze(1)

class BinaryMLP:
    def __init__(self, input_dim=2, hidden_dim=16, num_layers=1, activation="relu"):
        self.num_layers = num_layers
        self.activation = torch.relu if activation == "relu" else torch.tanh
        scale1 = (2.0 / input_dim) ** 0.5
        scale2 = (2.0 / hidden_dim) ** 0.5
        self.w1 = (torch.randn(input_dim, hidden_dim) * scale1).requires_grad_()
        self.b1 = torch.zeros(hidden_dim, requires_grad=True)
        if num_layers == 1:
            self.w2 = (torch.randn(hidden_dim, 1) * scale2).requires_grad_()
            self.b2 = torch.zeros(1, requires_grad=True)
            self.params = [self.w1, self.b1, self.w2, self.b2]
        else:
            self.w2 = (torch.randn(hidden_dim, hidden_dim) * scale2).requires_grad_()
            self.b2 = torch.zeros(hidden_dim, requires_grad=True)
            self.w3 = (torch.randn(hidden_dim, 1) * scale2).requires_grad_()
            self.b3 = torch.zeros(1, requires_grad=True)
            self.params = [self.w1, self.b1, self.w2, self.b2, self.w3, self.b3]

    def forward(self, X):
        h1 = self.activation(X @ self.w1 + self.b1)
        if self.num_layers == 1:
            return torch.sigmoid(h1 @ self.w2 + self.b2)
        h2 = self.activation(h1 @ self.w2 + self.b2)
        return torch.sigmoid(h2 @ self.w3 + self.b3)

    def cost(self, y_pred, y):
        eps = 1e-7
        y_pred = torch.clamp(y_pred, eps, 1.0 - eps)
        return -torch.mean(y * torch.log(y_pred) + (1.0 - y) * torch.log(1.0 - y_pred))

    def fit(self, X, y, epochs=200, lr=0.05, batch_size=32):
        n = X.shape[0]
        for epoch in range(epochs):
            indices = torch.randperm(n)
            for i in range(0, n, batch_size):
                idx = indices[i:i + batch_size]
                xb, yb = X[idx], y[idx]
                y_pred = self.forward(xb)
                loss = self.cost(y_pred, yb)
                loss.backward()
                with torch.no_grad():
                    for p in self.params:
                        p -= lr * p.grad
                        p.grad.zero_()

    def predict(self, X):
        with torch.no_grad():
            return (self.forward(X) >= 0.5).float()

_configs = [
    (1, 4, "relu"),
    (1, 8, "relu"),
    (1, 16, "relu"),
    (1, 32, "relu"),
    (2, 4, "relu"),
    (2, 8, "relu"),
    (2, 16, "relu"),
    (2, 32, "relu"),
    (1, 4, "tanh"),
    (1, 8, "tanh"),
    (1, 16, "tanh"),
    (1, 32, "tanh"),
    (2, 4, "tanh"),
    (2, 8, "tanh"),
    (2, 16, "tanh"),
    (2, 32, "tanh"),
]

_results = []
for _layers, _units, _act in _configs:
    torch.manual_seed(42)
    _model = BinaryMLP(input_dim=2, hidden_dim=_units, num_layers=_layers, activation=_act)
    _model.fit(_X_train_t, _y_train_t, epochs=200, lr=0.05, batch_size=32)
    _train_acc = (_model.predict(_X_train_t) == _y_train_t).float().mean().item()
    _test_acc = (_model.predict(_X_test_t) == _y_test_t).float().mean().item()
    _results.append({
        "hidden_layers": _layers,
        "hidden_units": _units,
        "activation": _act,
        "train_accuracy": round(_train_acc, 4),
        "holdout_accuracy": round(_test_acc, 4),
    })

binary_results_df = pd.DataFrame(_results)
print(binary_results_df)

Increasing the number of hidden units from 4 to 8 yields a sharp improvement in performance, moving accuracy from around 86 percent to 98 percent because 8 units provide enough capacity to separate the interlocking moon shapes. Expanding beyond 8 units to 16 and 32 units yields minimal additional benefit, as test accuracy plateaus around 98 to 99 percent.

Using two hidden layers instead of one provides only a marginal improvement once the network has at least 8 units, raising accuracy slightly from 98 percent to 99 percent. For smaller models with only 4 units, adding a second layer helps the network compute more complex compositional boundaries, particularly when using Tanh.

Both ReLU and Tanh achieve comparable top performance around 98 to 99 percent when given 8 or more units. At low capacity with only 4 units, Tanh outperforms ReLU (93.5 percent versus 86.0 percent) because its smooth, zero-centered sigmoid curve provides cleaner gradient flow and smoothly fits the curved boundary with fewer neurons.

## 3. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:
   - Label = 0 → [1, 0, 0]
   - Label = 1 → [0, 1, 0]
   - Label = 2 → [0, 0, 1]
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.

 - The **predicted class** corresponds to the unit with the highest probability.
   Example:
   - `[0.1, 0.3, 0.6] → class 2`
   - `[0.6, 0.2, 0.2] → class 0`

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [ ]:
from sklearn.datasets import make_classification
_X, _y = make_classification(n_samples=1000, n_classes=3, n_clusters_per_class=1, n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 4: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
_X, _y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 5. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

A common base class called BaseMLP can encapsulate the shared logic of parameter initialization, hidden layer transformations, and the stochastic gradient descent training loop. The base class manages the weight tensors, bias vectors, gradient zeroing, and backward step execution across mini-batches since this procedure is identical regardless of the task.

Task-specific classes such as BinaryMLP, MultiClassMLP, and RegressionMLP can inherit from BaseMLP and override only the output transformation and the loss function. BinaryMLP defines binary cross-entropy loss with sigmoid thresholding, MultiClassMLP defines categorical cross-entropy loss with softmax argmax class prediction, and RegressionMLP defines mean squared error loss with direct linear pass-through. This design eliminates duplicated training and layer-propagation code across all three models while isolating task-specific behavior to a few lines per subclass.

## 6. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE